<a href="https://colab.research.google.com/github/tomygoldboimel/public-opinion-simulation-analysis/blob/main/notebook1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Instalação de Dependências**

In [ ]:
!pip install pyreadstat google-generativeai shap joblib -q

**Imports e Configuração Geral**

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
import joblib

**Configurações do Experimento e Chave da API**

In [ ]:
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

LLM_PROVIDER = "gemini"
LLM_MODELS   = {
    "anthropic": "claude-sonnet-4-6",
    "openai":    "gpt-4o",
    "gemini":    "gemini-1.5-pro",
}
LLM_SAMPLE_N = 10
SAMPLE_SEED  = 42

os.makedirs("outputs", exist_ok=True)
print("Configurações OK.")

Configurações OK.


**Carregamento e Inspeção do Dataset**

In [ ]:
df_raw, meta = pyreadstat.read_sav("04839.sav", apply_value_formats=False)
print(f"{df_raw.shape[0]} respondentes × {df_raw.shape[1]} variáveis\n")

print("── COLUNAS (nome técnico → label) ──")
for col, label in zip(meta.column_names, meta.column_labels):
    print(f"  {col:<15} {label}")

print("\n── VALUE LABELS ──")
for i, (var, labels) in enumerate(meta.variable_value_labels.items()):
    print(f"  {var}: {labels}")
    if i >= 19: print("  ..."); break

print("\n── MISSING (%) ──")
pct = (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
print(pct[pct > 0].to_string())

2000 respondentes × 49 variáveis

── COLUNAS (nome técnico → label) ──
  IDADE           IDADE
  SEXO            SEXO
  ESCOLARIDADE    ESCOLARIDADE
  P1_1            P.01) Pensando no acesso e no atendimento dos diversos serviços presentes aqui na cidade, gostaria que dissesse em qual desses locais você acredita que existe mais diferença no tratamento de pessoas negras e pessoas brancas? (1º lugar)
  P1_2            P.01) Pensando no acesso e no atendimento dos diversos serviços presentes aqui na cidade, gostaria que dissesse em qual desses locais você acredita que existe mais diferença no tratamento de pessoas negras e pessoas brancas? (2º lugar)
  P1_3            P.01) Pensando no acesso e no atendimento dos diversos serviços presentes aqui na cidade, gostaria que dissesse em qual desses locais você acredita que existe mais diferença no tratamento de pessoas negras e pessoas brancas? (3º lugar)
  P2_1            P.02) Pensando no seu dia a dia aqui na cidade, em quais dos locais a s

**Mapeamento de Variáveis**

In [ ]:
FEATURE_COLS = {
    "SEXO":          "sexo",
    "IDADE":         "idade",
    "ESCOLARIDADE":  "escolaridade",
    "RACA_COR":      "raca_cor",
    "REND2":         "renda_familiar",
    "REGIAO":        "regiao",
}

TARGET_COLS = {
    "P6A": "abordagem_policial_racial",
    "P6B": "negros_universidades_positivo",
    "P6C": "representatividade_reduz_desigualdade",
    "P6D": "mercado_imobiliario_garante_moradia",
    "P6E": "clima_afeta_todos_igual",
}

QUESTION_TEXT = {
    "abordagem_policial_racial":
        "A abordagem policial é baseada no tipo de cabelo, tipo de vestimenta e cor de pele das pessoas.",
    "negros_universidades_positivo":
        "A maior presença de pessoas negras e indígenas nas universidades é bom para toda a sociedade.",
    "representatividade_reduz_desigualdade":
        "Aumentar a representatividade de pessoas negras, mulheres e população LGBTQIA+ na política e em cargos de poder contribui para diminuir as desigualdades estruturais.",
    "mercado_imobiliario_garante_moradia":
        "No Brasil, a construção de moradias pelo mercado imobiliário garante acesso à moradia digna para toda a população.",
    "clima_afeta_todos_igual":
        "As mudanças climáticas e eventos extremos atingem igualmente todas as pessoas, independente de cor ou classe social.",
}

SEXO_MAP  = {1: "masculino", 2: "feminino"}
REGIAO_MAP = {1: "Norte", 2: "Nordeste", 3: "Centro-Oeste", 4: "Sudeste", 5: "Sul"}
ESCOL_MAP = {
    1:  "analfabeto",
    2:  "sabe ler/escrever, sem escola",
    3:  "pré-escola",
    4:  "1ª série do fundamental",
    5:  "2ª série do fundamental",
    6:  "3ª série do fundamental",
    7:  "4ª série do fundamental (5º ano)",
    8:  "5ª série do fundamental (6º ano)",
    9:  "6ª série do fundamental",
    10: "7ª série do fundamental",
    11: "8ª série do fundamental (9º ano)",
    12: "1ª série do ensino médio",
    13: "2ª série do ensino médio",
    14: "3ª série do ensino médio (completo)",
    15: "superior incompleto",
    16: "superior completo",
}
RENDA_MAP = {
    1: "até 1 salário mínimo",
    2: "de 1 a 2 salários mínimos",
    3: "de 2 a 3 salários mínimos",
    4: "de 3 a 5 salários mínimos",
    5: "de 5 a 10 salários mínimos",
    6: "mais de 10 salários mínimos",
}
RACA_MAP = {1: "branca", 2: "parda", 3: "preta", 4: "amarela", 5: "indígena"}

LIKERT_SCALE = (
    "1 = Discordo totalmente | 2 = Discordo | "
    "3 = Nem concordo nem discordo | 4 = Concordo | 5 = Concordo totalmente"
)

FEATURE_NAMES = list(FEATURE_COLS.keys())
TARGET_NAMES  = list(TARGET_COLS.keys())

print("VARIÁVEIS:")
print(f"   Features : {FEATURE_NAMES}")
print(f"   Targets  : {TARGET_NAMES}")

VARIÁVEIS:
   Features : ['SEXO', 'IDADE', 'ESCOLARIDADE', 'RACA_COR', 'REND2', 'REGIAO']
   Targets  : ['P6A', 'P6B', 'P6C', 'P6D', 'P6E']


**Pré-processamento**

In [ ]:
cols_needed = FEATURE_NAMES + TARGET_NAMES
df = df_raw[cols_needed].copy()
df.rename(columns={**FEATURE_COLS, **TARGET_COLS}, inplace=True)

feature_names = list(FEATURE_COLS.values())
target_names  = list(TARGET_COLS.values())

MISSING_CODES = [99, 98, 97, 0, -1]
df = df.replace(MISSING_CODES, np.nan)

for col in ["renda_familiar", "escolaridade", "idade"]:
    df[col] = df[col].fillna(df[col].median())

for col in ["sexo", "raca_cor", "regiao"]:
    df[col] = df[col].fillna(df[col].mode()[0])

le = LabelEncoder()
for col in ["sexo", "raca_cor", "regiao"]:
    df[col] = le.fit_transform(df[col].astype(str))

X = df[feature_names].values

print(f"Feature matrix X: {X.shape}")
print(f"   Features: {feature_names}")

print("\n── Distribuição das variáveis-alvo ──")
for t in target_names:
    df[t] = df[t].where(df[t].between(1, 5), other=np.nan)
    vals = df[t].dropna()
    print(f"\n{t} | n={len(vals)} ({len(vals)/len(df)*100:.1f}% válidos)")
    print(vals.value_counts().sort_index().to_string())

Feature matrix X: (2000, 6)
   Features: ['sexo', 'idade', 'escolaridade', 'raca_cor', 'renda_familiar', 'regiao']

── Distribuição das variáveis-alvo ──

abordagem_policial_racial | n=1925 (96.2% válidos)
abordagem_policial_racial
1.0    808
2.0    439
3.0     90
4.0    216
5.0    372

negros_universidades_positivo | n=1965 (98.2% válidos)
negros_universidades_positivo
1.0    1437
2.0     277
3.0      87
4.0      81
5.0      83

representatividade_reduz_desigualdade | n=1912 (95.6% válidos)
representatividade_reduz_desigualdade
1.0    869
2.0    420
3.0    112
4.0    194
5.0    317

mercado_imobiliario_garante_moradia | n=1941 (97.0% válidos)
mercado_imobiliario_garante_moradia
1.0    510
2.0    383
3.0     97
4.0    378
5.0    573

clima_afeta_todos_igual | n=1945 (97.2% válidos)
clima_afeta_todos_igual
1.0    1094
2.0     312
3.0      62
4.0     186
5.0     291


**Random Forest com StratifiedKFold**

In [ ]:
def train_evaluate_rf(X, y_raw, target_name, n_splits=5):
    mask   = ~np.isnan(y_raw)
    Xc, yc = X[mask], y_raw[mask].astype(int)

    if len(np.unique(yc)) < 2:
        print(f"{target_name}: menos de 2 classes — pulando.")
        return {}

    rf = RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    y_pred = cross_val_predict(rf, Xc, yc, cv=cv)
    y_prob = cross_val_predict(rf, Xc, yc, cv=cv, method="predict_proba")
    rf.fit(Xc, yc)

    res = {
        "target":        target_name,
        "n_samples":     len(yc),
        "match_rate":    accuracy_score(yc, y_pred),
        "f1_macro":      f1_score(yc, y_pred, average="macro"),
        "f1_weighted":   f1_score(yc, y_pred, average="weighted"),
        "y_true":        yc,
        "y_pred_rf":     y_pred,
        "y_prob_rf":     y_prob,
        "indices_valid": np.where(mask)[0],
        "model":         rf,
    }

    print(f"\n── RF | {target_name} ──")
    print(f"   n={len(yc)} | match_rate={res['match_rate']:.3f} | F1-macro={res['f1_macro']:.3f}")
    print(classification_report(yc, y_pred, zero_division=0))
    return res


rf_results = {}
for t in target_names:
    rf_results[t] = train_evaluate_rf(X, df[t].values, t)


── RF | abordagem_policial_racial ──
   n=1925 | match_rate=0.262 | F1-macro=0.214
              precision    recall  f1-score   support

           1       0.46      0.34      0.39       808
           2       0.24      0.24      0.24       439
           3       0.07      0.19      0.11        90
           4       0.12      0.15      0.13       216
           5       0.20      0.21      0.21       372

    accuracy                           0.26      1925
   macro avg       0.22      0.22      0.21      1925
weighted avg       0.30      0.26      0.28      1925


── RF | negros_universidades_positivo ──
   n=1965 | match_rate=0.486 | F1-macro=0.232
              precision    recall  f1-score   support

           1       0.74      0.60      0.66      1437
           2       0.16      0.22      0.18       277
           3       0.12      0.25      0.16        87
           4       0.02      0.02      0.02        81
           5       0.10      0.18      0.13        83

    accuracy 

**Inferência LLM (Gemini)**

In [ ]:
import google.generativeai as genai

genai.configure(api_key=os.environ.get("GOOGLE_API_KEY", ""))
gemini_model = genai.GenerativeModel("gemini-2.0-flash")

def call_gemini(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            resp = gemini_model.generate_content(
                prompt,
                generation_config={"max_output_tokens": 10, "temperature": 0.0}
            )
            text = resp.text.strip()
            for char in text:
                if char in "12345":
                    return int(char)
            print(f"Resposta não parseável: '{text}'")
            return None
        except Exception as e:
            wait = 2 ** attempt
            print(f"  Erro (tentativa {attempt+1}): {e} — aguardando {wait}s")
            time.sleep(wait)
    return None


def build_persona_prompt(row, question_text):
    def decode(mapping, val, fallback="não informado"):
        try:    return mapping.get(int(val), fallback)
        except: return fallback

    sexo   = decode(SEXO_MAP,   row.get("sexo"))
    regiao = decode(REGIAO_MAP, row.get("regiao"))
    escol  = decode(ESCOL_MAP,  row.get("escolaridade"))
    renda  = decode(RENDA_MAP,  row.get("renda_familiar"))
    raca   = decode(RACA_MAP,   row.get("raca_cor"))
    idade  = int(row.get("idade", 35))

    return f"""Você é um(a) participante da pesquisa nacional brasileira \
"Percepção dos brasileiros sobre temas relacionados à desigualdade", realizada em 2023. \
Responda EXCLUSIVAMENTE a partir da perspectiva da pessoa descrita abaixo.

PERFIL:
- Sexo: {sexo}
- Idade: {idade} anos
- Região: {regiao}
- Escolaridade: {escol}
- Renda familiar mensal: {renda}
- Raça/cor (autodeclarada): {raca}

CONTEXTO (Brasil, 2023):
Alta informalidade no mercado de trabalho (~40%), forte desigualdade regional, \
debate intenso sobre cotas raciais, violência policial e representatividade política.

PERGUNTA:
"{question_text}"

INSTRUÇÃO:
Responda APENAS com um número inteiro de 1 a 5. Nenhuma outra palavra.
{LIKERT_SCALE}

Número (1–5):"""


def run_llm_inference(df, target_col, question_text,
                      n_samples=LLM_SAMPLE_N, seed=SAMPLE_SEED):
    valid  = df[df[target_col].between(1, 5)].copy()
    n      = min(n_samples, len(valid))
    sample = (
        valid
        .groupby(target_col, group_keys=False)
        .apply(lambda g: g.sample(frac=n / len(valid), random_state=seed))
        .head(n)
    )

    print(f"\n── Gemini | {target_col} ──")
    print(f"   n={len(sample)} | classes: {sample[target_col].value_counts().sort_index().to_dict()}")

    records = []
    for idx, row in tqdm(sample.iterrows(), total=len(sample), desc=target_col):
        prompt = build_persona_prompt(row[feature_names], question_text)
        pred   = call_gemini(prompt)
        records.append({"index_original": idx, "y_true": int(row[target_col]), "y_pred_llm": pred})
        time.sleep(6.0)

    results = pd.DataFrame(records)
    n_fail  = results["y_pred_llm"].isna().sum()
    if n_fail:
        print(f"{n_fail} respostas inválidas descartadas ({n_fail/len(results)*100:.1f}%)")
    return results.dropna(subset=["y_pred_llm"]).astype({"y_pred_llm": int})


llm_results = {}
for t in target_names:
    question       = QUESTION_TEXT.get(t, f"Pergunta sobre '{t}'")
    llm_results[t] = run_llm_inference(df, t, question)

**Métricas Comparativas**

In [ ]:
def compute_comparison(rf_res, llm_df, target_name):
    if not rf_res or llm_df.empty:
        return pd.DataFrame()

    common = set(rf_res["indices_valid"]) & set(llm_df["index_original"])
    if not common:
        print(f"{target_name}: sem índices comuns RF–LLM")
        return pd.DataFrame()

    mask_rf   = np.isin(rf_res["indices_valid"], list(common))
    idx_order = rf_res["indices_valid"][mask_rf]
    y_true    = rf_res["y_true"][mask_rf]
    y_rf      = rf_res["y_pred_rf"][mask_rf]

    llm_sub   = llm_df.set_index("index_original")
    valid_idx = [i for i in idx_order if i in llm_sub.index]

    y_true_al = np.array([y_true[k] for k, i in enumerate(idx_order) if i in llm_sub.index])
    y_rf_al   = np.array([y_rf[k]   for k, i in enumerate(idx_order) if i in llm_sub.index])
    y_llm     = np.array([llm_sub.loc[i, "y_pred_llm"] for i in valid_idx])

    rows = []
    for model_name, y_pred in [("Random Forest", y_rf_al), ("LLM (Gemini)", y_llm)]:
        rows.append({
            "target":      target_name,
            "modelo":      model_name,
            "n":           len(y_true_al),
            "match_rate":  accuracy_score(y_true_al, y_pred),
            "f1_macro":    f1_score(y_true_al, y_pred, average="macro",    zero_division=0),
            "f1_weighted": f1_score(y_true_al, y_pred, average="weighted", zero_division=0),
        })

    cmp = pd.DataFrame(rows)
    print(f"\n── Comparação | {target_name} ──")
    print(cmp.to_string(index=False))
    return cmp


comparisons = []
for t in target_names:
    cmp = compute_comparison(rf_results.get(t, {}), llm_results.get(t, pd.DataFrame()), t)
    if not cmp.empty:
        comparisons.append(cmp)

df_cmp = pd.concat(comparisons, ignore_index=True) if comparisons else pd.DataFrame()

print("\n" + "═"*60)
print("RESULTADO CONSOLIDADO — MARCO 1")
print("═"*60)
print(df_cmp.to_string(index=False) if not df_cmp.empty else "  Sem dados.")

**Visualização dos Resultados**

In [ ]:
def plot_comparison(df_cmp):
    if df_cmp.empty:
        print("Sem dados para plotar.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    COLORS = {"Random Forest": "#2ecc71", "LLM (Gemini)": "#3498db"}

    for ax, metric, title in zip(
        axes,
        ["match_rate", "f1_macro"],
        ["Match Rate (Acurácia Individual)", "F1-Macro"]
    ):
        pivot = df_cmp.pivot(index="target", columns="modelo", values=metric)
        pivot.plot(
            kind="bar", ax=ax,
            color=[COLORS.get(c, "#aaa") for c in pivot.columns],
            edgecolor="white", width=0.6
        )
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel("")
        ax.set_ylim(0, 1)
        ax.axhline(0.20, color="red", linestyle="--", alpha=0.5,
                   label="Baseline aleatório (0.20)")
        ax.legend(fontsize=8)
        ax.tick_params(axis="x", rotation=35)
        sns.despine(ax=ax)

    plt.suptitle(
        "Marco 1 — RF vs Gemini\n"
        "Percepção dos brasileiros sobre temas relacionados à desigualdade — CESOP 2023",
        fontsize=11, y=1.03
    )
    plt.tight_layout()
    plt.savefig("outputs/marco1_match_rate.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Gráfico salvo em outputs/marco1_match_rate.png")

plot_comparison(df_cmp)

**Exportação de Artefatos**

In [ ]:
df_cmp.to_csv("outputs/marco1_metrics.csv", index=False)
np.save("outputs/X_features.npy", X)

with open("outputs/feature_names.json", "w", encoding="utf-8") as f:
    json.dump(feature_names, f, ensure_ascii=False, indent=2)

for t in target_names:
    if rf_results.get(t):
        joblib.dump(rf_results[t]["model"], f"outputs/rf_model_{t}.joblib")

for t in target_names:
    if t in llm_results and not llm_results[t].empty:
        llm_results[t].to_csv(f"outputs/llm_preds_{t}.csv", index=False)

config = {
    "dataset":   "04839.sav — Percepção dos brasileiros sobre temas relacionados à desigualdade",
    "llm":       "gemini-1.5-pro",
    "sample_n":  LLM_SAMPLE_N,
    "targets":   TARGET_NAMES,
    "features":  feature_names,
}
with open("outputs/marco1_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Artefatos salvos em outputs/")
print("   rf_model_*.joblib  →  SHAP no Marco 2")
print("   llm_preds_*.csv    →  ablation no Marco 2")
print("   X_features.npy     →  reutilizado no Marco 2")

print("\n" + "═"*60)
print("CHECKLIST MARCO 1 CONCLUÍDO")
print("═"*60)
checks = [
    "04839.sav carregado (2000 respondentes × 49 variáveis)",
    "6 features sociodemográficas reais mapeadas",
    "5 variáveis-alvo P6A–P6E (escala Likert concordância)",
    "RF treinado com StratifiedKFold 5-fold",
    "Gemini chamado com template de persona anti-WEIRD",
    "Match rate calculado nos mesmos índices RF–Gemini",
    "Artefatos exportados para o Marco 2",
]
for c in checks:
    print(f"  [✓] {c}")